In [1]:
import chipwhisperer as cw
import os

scope = cw.scope()
target = cw.target(scope, cw.targets.CW305)

bitfile = r'C:\Users\sbista\ChipWhisperer\chipwhisperer\firmware\fpgas\aes\vivado\cw305_aes.runs\impl_100t\cw305_top.bit'
print("BIT exists?", os.path.exists(bitfile))

with open(bitfile, "rb") as f:
    target.fpga.FPGAProgram(f)

print("✅ FPGA programmed")


BIT exists? True
✅ FPGA programmed


In [2]:
print("Buildtime :", target.fpga_buildtime)
print("Crypt type:", target.fpga_read(target.REG_CRYPT_TYPE, 1))
print("Crypt rev :", target.fpga_read(target.REG_CRYPT_REV, 1))

Buildtime : 1/18/2026, 23:01
Crypt type: bytearray(b'\x02')
Crypt rev : bytearray(b'\x05')


In [3]:
from Crypto.Cipher import AES
import time

# NIST AES-128 test vector
key = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
pt  = bytes.fromhex("00112233445566778899aabbccddeeff")
exp = AES.new(key, AES.MODE_ECB).encrypt(pt)

# write KEY + PT + GO
target.fpga_write(target.REG_CRYPT_KEY, key)
target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.01)

ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

print("Expected:", exp.hex())
print("CW305 CT :", ct.hex())
print("MATCH?  :", ct == exp)


Expected: 69c4e0d86a7b0430d8cdb78070b4c55a
CW305 CT : 69c4e0d86a7b0430d8cdb78070b4c55a
MATCH?  : True


In [4]:
pt2 = bytes.fromhex("00112233445566778899aabbccddee00")

target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.01)
ct1 = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

target.fpga_write(target.REG_CRYPT_TEXTIN, pt2)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.01)
ct2 = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

print("CT1:", ct1.hex())
print("CT2:", ct2.hex())
print("Changed?", ct1 != ct2)


CT1: 69c4e0d86a7b0430d8cdb78070b4c55a
CT2: 7c99f42b6ee503309c6c1a67e97ac242
Changed? True


In [5]:
# try reversing key bytes
k_rev = key[::-1]
target.fpga_write(target.REG_CRYPT_KEY, k_rev)
target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.01)
ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
print("CT with key reversed:", ct.hex())
print("Match?", ct == exp)


CT with key reversed: f59d7cbf08fc47375511e6d9eecb6804
Match? False


In [6]:
from Crypto.Cipher import AES
import time

key = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
pt  = bytes.fromhex("00112233445566778899aabbccddeeff")
exp = AES.new(key, AES.MODE_ECB).encrypt(pt)

target.fpga_write(target.REG_CRYPT_KEY, key)
target.fpga_write(target.REG_CRYPT_TEXTIN, pt)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
time.sleep(0.02)
ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))

print("Expected:", exp.hex())
print("CW305 CT :", ct.hex())
print("MATCH?  :", ct == exp)


Expected: 69c4e0d86a7b0430d8cdb78070b4c55a
CW305 CT : 69c4e0d86a7b0430d8cdb78070b4c55a
MATCH?  : True


In [25]:
target.fpga_write(target.REG_CRYPT_KEY, key0)
target.fpga_write(target.REG_CRYPT_TEXTIN, pt0)
target.fpga_write(target.REG_CRYPT_GO, b"\x01")
ct = bytes(target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16))
print(ct.hex())


NameError: name 'key0' is not defined